# TASK 15 : Multi-Step AI Decision Agent

In [3]:
import os
import ollama
import gradio as gr

MODEL = "llama3.2:latest"
MEMORY_FILE = "DecisionMemory.txt"


def ollama_call(prompt):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]


def new_decision():
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        f.write("")

    return "✅ Previous conversation cleared. Start a new decision.", None


def decision_agent(user_input):

    if not os.path.exists(MEMORY_FILE):
        open(MEMORY_FILE, "w", encoding="utf-8").close()

    with open(MEMORY_FILE, "r", encoding="utf-8") as f:
        history = f.read()

    # First Question
    if history.strip() == "":

        prompt = f"""
You are an expert AI Decision Agent.

Decision Problem:
{user_input}

Analyze the problem step by step.

Generate:

1. Problem Understanding
2. Pros
3. Cons
4. Risk Analysis
5. Recommendation
6. Final Conclusion

Remember this conversation because the user will ask follow-up questions later.

Keep the response practical, professional and beginner-friendly.
"""

    # Follow-up Question
    else:

        prompt = f"""
You are an expert AI Decision Agent.

Previous Conversation:
{history}

New User Question:
{user_input}

Continue the conversation naturally.

Rules:
- Read the previous conversation carefully.
- Never repeat the previous report.
- Answer only the new question.
- Use previous context while giving recommendations.
- At the end remember this answer for future conversation.
"""

    answer = ollama_call(prompt)

    with open(MEMORY_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n\nUSER:\n{user_input}\n")
        f.write(f"\nAI:\n{answer}\n")

    return answer, MEMORY_FILE


with gr.Blocks(title="Memory Based AI Decision Agent") as demo:

    gr.Markdown("""
# 🧠 Memory Based AI Decision Agent

Ask your decision-related questions naturally.
The AI remembers the previous conversation until you click **New Decision**.
""")

    user_input = gr.Textbox(
        label="Ask Your Question",
        lines=4,
        placeholder="Example: Should I buy a new laptop?"
    )

    with gr.Row():
        ask_btn = gr.Button("Ask AI")
        new_btn = gr.Button("New Decision")

    output = gr.Textbox(
        label="AI Response",
        lines=20
    )

    file = gr.File(label="Download Decision Memory")

    ask_btn.click(
        decision_agent,
        inputs=user_input,
        outputs=[output, file]
    )

    new_btn.click(
        new_decision,
        outputs=[output, file]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
